In [1]:
!pip install -q -U transformers accelerate bitsandbytes datasets peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 39.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# 2. Authenticate Hugging Face
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(hf_token)

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 1. Paths (Matches your successful save directory)
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_path = "/kaggle/input/notebooks/dmshdissanayake/math-model-llm-1-5b-fine-tuning/math-solver-final-1.5B"

# 2. Load Tokenizer & Base
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    dtype=torch.float16,
    device_map={"": 0}
)

# 3. Load Specialist Adapter
ft_model = PeftModel.from_pretrained(base_model, adapter_path)
ft_model.eval()

def test_model_cot(model, question, label):
    # This system instruction triggers the CoT patterns learned from L3 data
    system_instruction = (
        "You are an engineering assistant. Solve the problem step-by-step. "
        "For each step, explain the logic, state the formula used, and show the calculation in LaTeX. "
        "Conclude with a 'Final Answer' section. In the end , explain the basic mathematical concepts needed."
    )
    
    # Updated prompt structure to include the system instruction
    prompt = f"### System:\n{system_instruction}\n\n### Question:\n{question}\n\n### Answer:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=1024, # Increased to allow for longer CoT steps
            do_sample=False, 
            pad_token_id=tokenizer.eos_token_id
        )
    
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True).split("### Answer:\n")[-1]
    print(f"\n{'='*20} {label} {'='*20}\n{answer.strip()}")

def test_model(model, question, label):
    prompt = f"### Question:\n{question}\n\n### Answer:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=512,
            do_sample=False, 
            pad_token_id=tokenizer.eos_token_id
        )
    
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True).split("### Answer:\n")[-1]
    print(f"\n{'='*20} {label} {'='*20}\n{answer.strip()}")

# 4. The Engineering Stress Test
# This tests the "Refined Math" logic learned from the UltraData-Math L3 tier
test_q = "A spherical tank has a radius of 3m and is half-full of water. Calculate the volume of water in Liters using LaTeX for all steps. (Use pi = 3.14159)"

test_model(base_model, test_q, "BASE MODEL")
test_model_cot(ft_model, test_q, "FINE-TUNED MODEL")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



==================== BASE MODEL ====================
To calculate the volume of water in the tank, we first need to find the total volume of the sphere and then divide it by two since the tank is only half full.

The formula for the volume \( V \) of a sphere is:

\[ V = \frac{4}{3} \pi r^3 \]

Given that the radius \( r \) is 3 meters, we substitute this value into the formula:

\[ V = \frac{4}{3} \times 3.14159 \times (3)^3 \]

First, calculate \( 3^3 \):

\[ 3^3 = 27 \]

Now multiply:

\[ V = \frac{4}{3} \times 3.14159 \times 27 \]

\[ V = \frac{4}{3} \times 84.78463 \]

\[ V = 115.90752 \text{ cubic meters} \]

Since 1 cubic meter equals 1000 liters, convert the volume from cubic meters to liters:

\[ V_{\text{liters}} = 115.90752 \times 1000 \]

\[ V_{\text{liters}} = 115907.52 \text{ liters} \]

Thus, the volume of water in the tank is approximately **115907.52 liters**.

==================== FINE-TUNED MODEL ====================
#### Step 1: Understand the Problem
The goal is t

In [4]:
def generate_fix(model, question):
    # Triggers the refined reasoning path learned from UltraData-Math L3
    prompt = (
        f"### Instruction:\n"
        f"Solve the following problem with engineering-grade precision. "
        f"Show every step using LaTeX and verify the 'half-full' constraint."
        f"pi = 3.14159\n\n"
        f"### Question:\n{question}\n\n"
        f"### Answer:\n"
    )
    
    # Move inputs to the GPU
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    # Generate the response with a high token limit to allow full CoT steps
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=1024, 
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode and extract only the generated answer
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = full_output.split("### Answer:\n")[-1].strip()
    return answer

In [5]:
# Define your challenging math question
test_q = "A spherical tank has a radius of 3m and is half-full of water. Calculate the volume of water in Liters using LaTeX for all steps. (Use pi = 3.14159)"

# Run the fix function
refined_answer = generate_fix(ft_model, test_q)

# Display the results
print("--- Refined Engineering Solution ---")
print(refined_answer)

--- Refined Engineering Solution ---
To solve this, we first calculate the total volume of the sphere.

The formula for the volume \( V \) of a sphere is:

\[ V = \frac{4}{3} \pi r^3 \]

Given that the radius \( r \) is 3 meters, substitute into the formula:

\[ V = \frac{4}{3} \times 3.14159 \times (3)^3 \]

First compute \( 3^3 \):

\[ 3^3 = 27 \]

Now multiply:

\[ V = \frac{4}{3} \times 3.14159 \times 27 \]

Next, perform multiplication:

\[ \frac{4}{3} \times 86.0133 = 114.6813 \]

Thus, the total volume of the sphere is approximately **114.6813 cubic meters**.

Since the tank is only half full, divide by 2 to find the volume of water:

\[ \text{Volume of water} = \frac{114.6813}{2} = 57.34065 \] cubic meters.

Convert cubic meters to liters: 

\[ 1 \text{ m}^3 = 1000 \text{ L} \]

So,

\[ 57.34065 \text{ m}^3 \times 1000 = 57340.65 \text{ L} \]

Therefore, the volume of water in the tank is **57340.65 liters**. This follows from the 'half-full' constraint, ensuring the answer ali